# Week 4 — TD Learning, SARSA, Q-learning, Expected SARSA · Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Use one transition to compare three algorithms. Start with **Exercises 3, 4 and 5** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-use-one-transition-to-compare-three-algorithms).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
import _pkg_loader  # registers w04_project
from w04_project import agents, env_utils, solutions
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## 1. TD(0) prediction (one star)
Implement the one-step TD(0) update
$$V(s_t) \leftarrow V(s_t) + \alpha \bigl[r_{t+1} + \gamma V(s_{t+1}) - V(s_t)\bigr]$$
as a pure function applied along a list of `(s, r, s')` transitions. A terminal next state is encoded as `s' = -1` (do not bootstrap).

> **Hint** (try yourself first): copy V; loop; for terminals zero out the bootstrap term.

In [ ]:
def td0_update(traj, V, alpha=0.1, gamma=1.0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(solutions.exercise_1_td0, td0_update)([(0, 1.0, 1), (1, 2.0, -1)], np.zeros(4), alpha=1.0, gamma=1.0)
assert ref[0] == 1.0 and ref[1] == 2.0
print('exercise 1 OK')

## 2. n-step return (two stars)
Compute $G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$. If the trajectory ends inside the window (len(rewards) < n), drop the bootstrap term.

> **Hint**: two cases. (i) len(rewards) >= n: discounted sum of first n rewards + bootstrap. (ii) shorter: full discounted return, no bootstrap.

In [ ]:
def nstep_return(rewards, V_terminal, gamma, n):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert select_exercise(solutions.exercise_2_nstep_return, nstep_return)([1, 2, 3], 10.0, 1.0, 2) == 13.0
assert select_exercise(solutions.exercise_2_nstep_return, nstep_return)([1, 2], 10.0, 1.0, 5) == 3.0
print('exercise 2 OK')

## 3. SARSA single-step update (one star)
Apply the on-policy update $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma Q(s',a') - Q(s,a)]$. When `done`, zero the bootstrap term.

In [ ]:
def sarsa_step(Q, s, a, r, sp, ap, done, alpha, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
Q = np.zeros((3, 2))
Qn = select_exercise(solutions.exercise_3_sarsa_step, sarsa_step)(Q, 0, 1, 1.0, 1, 0, False, 0.5, 1.0)
assert Qn[0, 1] == 0.5
print('exercise 3 OK')

## 4. Q-learning single-step update (one star)
Apply the off-policy update $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$.

In [ ]:
def qlearn_step(Q, s, a, r, sp, done, alpha, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
Q = np.array([[0.0, 0.0], [3.0, 5.0]])
Qn = select_exercise(solutions.exercise_4_qlearn_step, qlearn_step)(Q, 0, 1, 1.0, 1, False, 0.5, 1.0)
assert Qn[0, 1] == 3.0
print('exercise 4 OK')

## 5. Expected-SARSA target (two stars)
Given $Q(s',\cdot)$ and the behaviour-policy mixing param $\varepsilon$ (epsilon-greedy), compute the *expected* bootstrap term $\mathbb{E}_{a'\sim\pi}[Q(s',a')]$.
Verify it equals $\max Q$ when $\varepsilon=0$ and the *mean* when $\varepsilon=1$.

In [ ]:
def expected_sarsa_target(Q_sp, eps):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert select_exercise(solutions.exercise_5_expected_sarsa_target, expected_sarsa_target)(np.array([1, 5, 2.]), 0.0) == 5.0
assert abs(select_exercise(solutions.exercise_5_expected_sarsa_target, expected_sarsa_target)(np.array([1, 5, 2.]), 1.0) - 8/3) < 1e-9
print('exercise 5 OK')

## 6. Double Q-learning step (two stars)
Maintain two estimates $Q_1, Q_2$. On each step flip a fair coin: update $Q_1$ using $Q_2$ to evaluate the argmax of $Q_1$, and symmetrically for $Q_2$.

> **Hint**: this *decouples* selection and evaluation, reducing coupled selection/evaluation error. The learned tables need not be independent or exactly unbiased.

In [ ]:
def double_q_step(Q1, Q2, s, a, r, sp, done, alpha, gamma, update_first):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
Q1 = np.zeros((2, 2)); Q2 = np.array([[0., 0.], [3., 7.]])
Q1n, _ = select_exercise(solutions.exercise_6_double_q_step, double_q_step)(Q1, Q2, 0, 1, 0.0, 1, False, 1.0, 1.0, True)
assert Q1n[0, 1] == 3.0
print('exercise 6 OK')

## 7. Empirical TD vs MC variance (two stars)
Generate `n_episodes` rollouts where the per-step reward is i.i.d. $\mathcal{N}(0,\sigma^2)$ and assume we *know* $V^\pi(s_0) = 0$. Compute the variance of (i) the TD(0) target $r_0 + V^\pi(s_1)$ and (ii) the Monte-Carlo return $G_0$ for the same trajectories. TD variance is independent of horizon; MC variance grows linearly.

In [ ]:
def td_vs_mc_variance(rewards_per_episode, V_true, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
eps_data = [list(rng.normal(0, 1, size=10)) for _ in range(500)]
v_td, v_mc = select_exercise(solutions.exercise_7_td_error_var, td_vs_mc_variance)(eps_data, V_true=0.0, gamma=1.0)
assert v_td < v_mc and 0.5 < v_td < 1.5 and 7.0 < v_mc < 13.0
print(f'TD var ~ {v_td:.2f}, MC var ~ {v_mc:.2f}  OK')

## 8. CliffWalking bake-off (two stars)
Run SARSA, Q-learning, and Expected SARSA on `CliffWalking` for `n_episodes` over `n_seeds`. Plot smoothed mean curves with a shaded band. Observation to make: SARSA learns a safer longer path; Q-learning learns the optimal greedy path next to the cliff but with worse exploration-time returns.

In [ ]:
def bakeoff(make_env, n_episodes=300, n_seeds=3, alpha=0.5, eps_decay=150):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
out = select_exercise(solutions.exercise_8_bakeoff, bakeoff)(env_utils.make_cliffwalking, n_episodes=150, n_seeds=2, eps_decay=80)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for name, color in zip(['sarsa', 'q_learning', 'expected_sarsa'], [RL_PALETTE['blue'], RL_PALETTE['orange'], RL_PALETTE['teal']]):
    m = out[name].mean(0); s = out[name].std(0)
    x = np.arange(len(m))
    ax.plot(x, m, label=name, color=color)
    ax.fill_between(x, m - s, m + s, alpha=0.2, color=color)
ax.set_xlabel('episode'); ax.set_ylabel('return'); ax.legend(); ax.set_ylim(bottom=-200)
fig.tight_layout(); plt.show()
print('exercise 8 OK')

## 9. Off-policy lead scoring with SNIPS (three stars — meaningful objective)
**Matching application.** A synthetic insurance-style lead router currently sends leads to brokers under an $\varepsilon$-greedy behaviour policy $\pi_b$. We have a log of `(features, broker, conversion)` tuples plus the per-tuple $\pi_b(\text{broker}\,|\,\text{features})$. We want the *target-policy* value under a new TD-trained policy $\pi_t$ before we deploy it.

Implement the **self-normalised importance sampling** estimator
$$\hat V_{\text{SNIPS}}(\pi_t) = \frac{\sum_i w_i r_i}{\sum_i w_i}, \qquad w_i = \pi_t(a_i|x_i) / \pi_b(a_i|x_i).$$
This is the work-horse off-policy evaluator we will return to in Week 11.

> **Hint**: clip the denominator at 1e-12 to avoid blow-up. SNIPS is generally biased in finite samples and consistent under coverage and moment assumptions; lower variance than IPS is not universal. A zero total target weight means the estimate is unsupported, not zero-valued.

In [ ]:
def snips(rewards, behaviour_probs, target_probs):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0); N = 5000
behaviour_probs = np.full(N, 0.5); target_probs = np.full(N, 0.25)
arms = rng.integers(0, 2, size=N); rewards = (arms == 0).astype(float) * 2.0
est = select_exercise(solutions.exercise_9_snips, snips)(rewards, behaviour_probs, target_probs)
assert 0.7 < est < 1.3
print(f'SNIPS estimate = {est:.3f}  OK')

## Reflection
1. Why does SARSA learn a safer policy than Q-learning, even though both use the same $\varepsilon$-greedy behaviour?
2. Under what conditions does Q-learning's max-bias actually hurt? Construct a 2-state toy MDP where Double Q-learning is provably better in expectation.
3. If you had a single hyper-parameter knob, would you increase n (return horizon) or decrease $\alpha$ (step size) to reduce variance? Reason through bias and convergence rate.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>1. TD(0) prediction (one star)</summary>

> **Hint** (try yourself first): copy V; loop; for terminals zero out the bootstrap term.

</details>

<details>
<summary>2. n-step return (two stars)</summary>

> **Hint**: two cases. (i) len(rewards) >= n: discounted sum of first n rewards + bootstrap. (ii) shorter: full discounted return, no bootstrap.

</details>

<details>
<summary>3. SARSA single-step update (one star)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>4. Q-learning single-step update (one star)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>5. Expected-SARSA target (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>6. Double Q-learning step (two stars)</summary>

> **Hint**: this *decouples* selection and evaluation, reducing coupled selection/evaluation error. The learned tables need not be independent or exactly unbiased.

</details>

<details>
<summary>7. Empirical TD vs MC variance (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>8. CliffWalking bake-off (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>9. Off-policy lead scoring with SNIPS (three stars — meaningful objective)</summary>

> **Hint**: clip the denominator at 1e-12 to avoid blow-up. SNIPS is generally biased in finite samples and consistent under coverage and moment assumptions; lower variance than IPS is not universal. A zero total target weight means the estimate is unsupported, not zero-valued.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — The maximum of unbiased estimates can be biased

**Goal.** Locate the selection step that biases a maximum of noisy action values. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** Four actions truly have value zero. Each estimated action value has unbiased
normal noise. Predict E[max Q̂] and compare it with evaluating the selected
action using an independent noisy estimate. This isolates the selection
mechanism behind maximization bias; it is not a universal performance claim
about two trained Q-learning agents.

**Work a small case by hand.** With two tied actions and independent standard-normal errors Z₁,Z₂,
max(Z₁,Z₂)=(Z₁+Z₂+|Z₁-Z₂|)/2. Since Z₁-Z₂ has variance 2,
E|Z₁-Z₂|=2/sqrt(pi), giving E[max]=1/sqrt(pi)≈0.5642 although each
estimate has mean zero. If an independent estimate evaluates the chosen
action, its conditional expected error is zero, whatever action was selected.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/02-expectation-variance.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
rng_stats = np.random.default_rng(104)
selection_values = rng_stats.normal(size=(4000,4))
evaluation_values = rng_stats.normal(size=(4000,4))
chosen = selection_values.argmax(axis=1)
max_targets = selection_values.max(axis=1)
double_targets = evaluation_values[np.arange(4000), chosen]
print('True maximum value = 0')
print('Mean max noisy estimate:',max_targets.mean())
print('Independent estimate of selected action:',double_targets.mean())
fig, axes = plt.subplots(1,2,figsize=(9,3.3))
shared_bins=np.linspace(min(max_targets.min(),double_targets.min()),max(max_targets.max(),double_targets.max()),36)
axes[0].hist(max_targets,bins=shared_bins,alpha=.6,label='Reuse selected estimate')
axes[0].hist(double_targets,bins=shared_bins,alpha=.5,label='Independent evaluation estimate')
axes[0].axvline(0,color=RL_PALETTE['red'],ls='--')
axes[0].set(xlabel='Bootstrap target without reward/discount',ylabel='Independent repetitions')
axes[0].legend(fontsize=8)
counts=[1,2,4,8,16]
means=[rng_stats.normal(size=(3000,k)).max(axis=1).mean() for k in counts]
axes[1].plot(counts,means,'o-'); axes[1].set(xlabel='Number of tied actions',ylabel='Observed maximization bias')
fig.tight_layout(); plt.show()


**Interpret and transfer.** Selection favors positive errors. Independent evaluation breaks that
particular coupling under this fixture. In practice two networks can have
correlated errors, and Double Q can introduce other biases.

**Transfer.** Replace the independent evaluation noise by 0.8 times the
selection noise plus independent noise. Predict the direction of the residual
bias, then measure it.


**Read your result.** Compare the positive center of reused maxima with the roughly zero-centered
independent evaluations. The right panel changes the number of tied actions;
it isolates a selection effect rather than a learned-policy performance ranking.

<details><summary>Worked feedback for the changed assumption</summary>

Let the evaluation error for each action be 0.8Z+0.6E, with E independent
standard normal. It still has variance 1, but conditional on selection its
mean is 0.8 times the selected Z. The expected selected error is therefore
0.8E[max Z], so correlation restores positive bias. Independent networks in
name need not have independent errors in practice.

</details>
